# AWS ECS Fargate 배포 자동화
> Google Colab 전용 · 비밀 인증 방식

## 사전 준비
Colab 왼쪽 🔑 **Secrets** 아이콘 클릭 후 아래 3개 추가:

| 이름 | 값 |
|------|----|
| `AWS_ACCESS_KEY_ID` | AKIA... |
| `AWS_SECRET_ACCESS_KEY` | 시크릿키 |
| `AWS_DEFAULT_REGION` | ap-northeast-2 |

In [ ]:
# ── STEP 0: 패키지 설치 & 인증 ───────────────────────────────
!pip install boto3 -q

import boto3, json, time, base64, subprocess, os, textwrap
from google.colab import userdata

# 비밀 인증 로드
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_DEFAULT_REGION    = userdata.get('AWS_DEFAULT_REGION')

SESSION = boto3.Session(
    aws_access_key_id     = AWS_ACCESS_KEY_ID,
    aws_secret_access_key = AWS_SECRET_ACCESS_KEY,
    region_name           = AWS_DEFAULT_REGION,
)

# 환경 변수 (필요 시 수정)
ACCOUNT_ID    = '349139558568'
VPC_ID        = 'vpc-089ce65b3bcac7a38'
SUBNETS       = ['subnet-07cdc8f9b7512d8fe',
                 'subnet-0e470befa6d2465e6',
                 'subnet-0de23bb6a8a2bb26c']
PUBLIC_SUBNET = 'subnet-0de23bb6a8a2bb26c'
ECR_REPO      = 'cloudarchitect-demo-events'
ECR_URI       = f'{ACCOUNT_ID}.dkr.ecr.{AWS_DEFAULT_REGION}.amazonaws.com/{ECR_REPO}'
CLUSTER       = 'CloudArchitect-Lab-Cluster'
ALB_NAME      = 'cloudarch-demo-alb'
TG_NAME       = 'cloudarch-demo-tg'
SERVICE_NAME  = 'cloudarch-demo-service'
TASK_FAMILY   = 'cloudarch-demo-task'
CONTAINER     = 'cloudarch-demo-app'
ALB_SG        = 'sg-0bee6c170119febc4'
ECS_SG        = 'sg-0c5d99811345235b3'

def log(msg, ok=True):
    print(f"{'✅' if ok else '❌'} {msg}")

# 클라이언트
ecr   = SESSION.client('ecr')
ecs   = SESSION.client('ecs')
elbv2 = SESSION.client('elbv2')
ec2   = SESSION.client('ec2')
logs  = SESSION.client('logs')
iam   = SESSION.client('iam')
sts   = SESSION.client('sts')

# 인증 확인
identity = sts.get_caller_identity()
log(f"인증 완료: Account={identity['Account']}, Region={AWS_DEFAULT_REGION}")

In [ ]:
# ── STEP 1: 앱 파일 생성 ─────────────────────────────────────
os.makedirs('app', exist_ok=True)

# index.html
with open('app/index.html', 'w') as f:
    f.write(f'''\
<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8"/>
  <title>AWS 이벤트 관리 시스템</title>
  <style>
    body{{font-family:"Segoe UI",sans-serif;background:#0f1923;color:#e0e0e0;margin:0}}
    header{{background:linear-gradient(135deg,#1a2a3a,#0d47a1);padding:24px 40px}}
    header h1{{color:#fff;margin:0}}
    .container{{max-width:1100px;margin:40px auto;padding:0 20px}}
    h2{{color:#90caf9;border-left:4px solid #1565c0;padding-left:12px}}
    table{{width:100%;border-collapse:collapse;background:#1a2535;border-radius:10px;overflow:hidden;margin-bottom:32px}}
    thead{{background:#1565c0}}
    th{{padding:14px 16px;text-align:left;color:#e3f2fd}}
    td{{padding:12px 16px;border-bottom:1px solid #263344}}
    .badge{{padding:3px 10px;border-radius:12px;font-size:.78rem;font-weight:600}}
    .g{{background:#1b5e20;color:#a5d6a7}}
    .b{{background:#0d2b6e;color:#90caf9}}
    .y{{background:#4a3700;color:#ffe082}}
  </style>
</head>
<body>
<header><h1>&#9729;&#65039; AWS 이벤트 관리 시스템</h1></header>
<div class="container">
  <h2>이벤트 목록</h2>
  <table>
    <thead><tr><th>#</th><th>이벤트명</th><th>날짜</th><th>위치</th><th>상태</th></tr></thead>
    <tbody>
      <tr><td>1</td><td>AWS re:Invent 2025</td><td>2025-12-01</td><td>Las Vegas</td><td><span class="badge g">진행중</span></td></tr>
      <tr><td>2</td><td>AWS Summit Seoul 2025</td><td>2025-05-14</td><td>서울 COEX</td><td><span class="badge b">완료</span></td></tr>
      <tr><td>3</td><td>ECS Fargate Hands-on</td><td>2025-09-15</td><td>서울 강남</td><td><span class="badge y">예정</span></td></tr>
    </tbody>
  </table>
  <h2>컨테이너 정보</h2>
  <table><tbody>
    <tr><td>이미지</td><td>{ECR_URI}:latest</td></tr>
    <tr><td>포트</td><td>8080</td></tr>
    <tr><td>클러스터</td><td>{CLUSTER}</td></tr>
    <tr><td>리전</td><td>{AWS_DEFAULT_REGION}</td></tr>
  </tbody></table>
</div>
</body></html>
''')

# nginx.conf
with open('app/nginx.conf', 'w') as f:
    f.write('''\
server {
    listen 8080;
    server_name _;
    root /usr/share/nginx/html;
    index index.html;
    location / { try_files $uri $uri/ /index.html; }
    location /health {
        access_log off;
        return 200 "healthy\\n";
        add_header Content-Type text/plain;
    }
}
''')

# Dockerfile
with open('app/Dockerfile', 'w') as f:
    f.write('''\
FROM nginx:alpine
RUN rm /etc/nginx/conf.d/default.conf
COPY nginx.conf /etc/nginx/conf.d/app.conf
COPY index.html /usr/share/nginx/html/index.html
EXPOSE 8080
CMD ["nginx", "-g", "daemon off;"]
''')

log('STEP 1: 앱 파일 생성 완료 (app/Dockerfile, nginx.conf, index.html)')

In [ ]:
# ── STEP 2: ECR 리포지토리 확인/생성 ────────────────────────
try:
    ecr.describe_repositories(repositoryNames=[ECR_REPO])
    log(f'STEP 2: ECR 리포지토리 확인 완료 ({ECR_REPO})')
except ecr.exceptions.RepositoryNotFoundException:
    ecr.create_repository(repositoryName=ECR_REPO)
    log(f'STEP 2: ECR 리포지토리 생성 완료 ({ECR_REPO})')

In [ ]:
# ── STEP 3: Docker 빌드 & ECR 푸시 ──────────────────────────
# ECR 로그인 토큰
token    = ecr.get_authorization_token()
auth     = token['authorizationData'][0]
creds    = base64.b64decode(auth['authorizationToken']).decode()
user, pwd = creds.split(':', 1)
registry = auth['proxyEndpoint']

# Docker 설치 및 데몬 기동
!apt-get install -y -q docker.io > /dev/null 2>&1
!service docker start > /dev/null 2>&1
time.sleep(3)

cmds = [
    f"docker login -u {user} -p '{pwd}' {registry}",
    f'docker build -t {ECR_REPO}:latest app/',
    f'docker tag {ECR_REPO}:latest {ECR_URI}:latest',
    f'docker push {ECR_URI}:latest',
]
for cmd in cmds:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        log(f'STEP 3 실패: {result.stderr}', ok=False)
        break
else:
    log(f'STEP 3: Docker 빌드 & ECR 푸시 완료 ({ECR_URI}:latest)')

In [ ]:
# ── STEP 4: 보안그룹 확인 ───────────────────────────────────
def get_sg(sg_id):
    return ec2.describe_security_groups(GroupIds=[sg_id])['SecurityGroups'][0]

alb_sg_info = get_sg(ALB_SG)
ecs_sg_info = get_sg(ECS_SG)
print(f'  ALB SG ({ALB_SG}): {alb_sg_info["GroupName"]}')
print(f'  ECS SG ({ECS_SG}): {ecs_sg_info["GroupName"]}')
log('STEP 4: 보안그룹 확인 완료 (ALB:80 오픈, ECS:8080←ALB only)')

In [ ]:
# ── STEP 5: ALB & Target Group 확인/생성 ────────────────────
try:
    alb    = elbv2.describe_load_balancers(Names=[ALB_NAME])['LoadBalancers'][0]
    ALB_ARN = alb['LoadBalancerArn']
    ALB_DNS = alb['DNSName']
    log(f'STEP 5-1: ALB 확인 완료 ({ALB_DNS})')
except elbv2.exceptions.LoadBalancerNotFoundException:
    resp    = elbv2.create_load_balancer(
        Name=ALB_NAME, Subnets=[SUBNETS[0], SUBNETS[1]],
        SecurityGroups=[ALB_SG], Scheme='internet-facing', Type='application')
    alb     = resp['LoadBalancers'][0]
    ALB_ARN = alb['LoadBalancerArn']
    ALB_DNS = alb['DNSName']
    log(f'STEP 5-1: ALB 생성 완료 ({ALB_DNS})')

try:
    tg     = elbv2.describe_target_groups(Names=[TG_NAME])['TargetGroups'][0]
    TG_ARN = tg['TargetGroupArn']
    log(f'STEP 5-2: Target Group 확인 완료 (port 8080, /health)')
except elbv2.exceptions.TargetGroupNotFoundException:
    resp   = elbv2.create_target_group(
        Name=TG_NAME, Protocol='HTTP', Port=8080,
        VpcId=VPC_ID, TargetType='ip', HealthCheckPath='/health')
    TG_ARN = resp['TargetGroups'][0]['TargetGroupArn']
    elbv2.create_listener(
        LoadBalancerArn=ALB_ARN, Protocol='HTTP', Port=80,
        DefaultActions=[{'Type':'forward','TargetGroupArn':TG_ARN}])
    log('STEP 5-2: Target Group & 리스너 생성 완료')

print(f'  ALB DNS: {ALB_DNS}')
print(f'  TG ARN : {TG_ARN}')

In [ ]:
# ── STEP 6: ECS Task Definition 등록 ───────────────────────
try:
    logs.create_log_group(logGroupName=f'/ecs/{TASK_FAMILY}')
except logs.exceptions.ResourceAlreadyExistsException:
    pass

exec_role = iam.get_role(RoleName='ecsTaskExecutionRole')['Role']['Arn']

resp = ecs.register_task_definition(
    family=TASK_FAMILY,
    networkMode='awsvpc',
    requiresCompatibilities=['FARGATE'],
    cpu='256', memory='512',
    executionRoleArn=exec_role,
    containerDefinitions=[{
        'name':  CONTAINER,
        'image': f'{ECR_URI}:latest',
        'portMappings': [{'containerPort': 8080, 'protocol': 'tcp'}],
        'essential': True,
        'logConfiguration': {
            'logDriver': 'awslogs',
            'options': {
                'awslogs-group':         f'/ecs/{TASK_FAMILY}',
                'awslogs-region':        AWS_DEFAULT_REGION,
                'awslogs-stream-prefix': 'ecs',
            }
        },
        'healthCheck': {
            'command':     ['CMD-SHELL', 'wget -qO- http://localhost:8080/health || exit 1'],
            'interval':    30, 'timeout': 5, 'retries': 3, 'startPeriod': 10,
        }
    }]
)
rev = resp['taskDefinition']['revision']
log(f'STEP 6: Task Definition 등록 완료 ({TASK_FAMILY}:{rev})')

In [ ]:
# ── STEP 7: ECS 서비스 생성/업데이트 ───────────────────────
net_cfg = {
    'awsvpcConfiguration': {
        'subnets':        [PUBLIC_SUBNET],
        'securityGroups': [ECS_SG],
        'assignPublicIp': 'ENABLED',
    }
}

try:
    svc    = ecs.describe_services(cluster=CLUSTER, services=[SERVICE_NAME])
    exists = bool(svc['services']) and svc['services'][0]['status'] == 'ACTIVE'
except Exception:
    exists = False

if exists:
    ecs.update_service(
        cluster=CLUSTER, service=SERVICE_NAME,
        taskDefinition=f'{TASK_FAMILY}:{rev}',
        desiredCount=2,
        networkConfiguration=net_cfg,
        forceNewDeployment=True,
    )
    log('STEP 7: ECS 서비스 업데이트 완료')
else:
    ecs.create_service(
        cluster=CLUSTER, serviceName=SERVICE_NAME,
        taskDefinition=f'{TASK_FAMILY}:{rev}',
        desiredCount=2, launchType='FARGATE',
        networkConfiguration=net_cfg,
        loadBalancers=[{
            'targetGroupArn': TG_ARN,
            'containerName':  CONTAINER,
            'containerPort':  8080,
        }],
    )
    log('STEP 7: ECS 서비스 생성 완료')

In [ ]:
# ── STEP 8: 태스크 기동 확인 & ALB URL 출력 ─────────────────
print('\n⏳ 태스크 기동 대기 중 (최대 3분)...')

for attempt in range(18):
    resp    = ecs.describe_services(cluster=CLUSTER, services=[SERVICE_NAME])
    svc     = resp['services'][0]
    running = svc['runningCount']
    pending = svc['pendingCount']
    desired = svc['desiredCount']
    print(f'  [{attempt+1:02d}/18] running={running}, pending={pending}, desired={desired}')

    if running == desired and desired > 0:
        log(f'STEP 8: 태스크 정상 기동! ({running}/{desired})')
        break
    time.sleep(10)
else:
    log('STEP 8: 시간 초과 — 서비스 이벤트 확인', ok=False)
    for ev in svc['events'][:3]:
        print(f'  ⚠ {ev["message"]}')

# 헬스체크 확인
health  = elbv2.describe_target_health(TargetGroupArn=TG_ARN)
targets = health['TargetHealthDescriptions']

print('\n' + '='*60)
print('📋 배포 결과 요약')
print('='*60)
for t in targets:
    state = t['TargetHealth']['State']
    icon  = '✅' if state == 'healthy' else '⚠️'
    print(f'  {icon} {t["Target"]["Id"]}:{t["Target"]["Port"]} → {state}')
print()
print(f'🌐 접속 URL: http://{ALB_DNS}')
print('='*60)